In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [9]:
print(len(files))

72


In [10]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [11]:
len(documents)

72

In [19]:
print(documents[0])

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [16]:
print(documents[0].keys())

dict_keys(['content', 'filename'])


In [20]:
print(documents[0].keys())
print(documents[0]["filename"])
print(documents[0]["content"][:211])

dict_keys(['content', 'filename'])
01-agentic-rag/lessons/01-intro.md
# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system 


In [21]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [22]:
query = "How does the agentic loop keep calling the model until it stops?"

In [23]:
results = index.search(
    query=query,
    num_results=5
)

print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [27]:
def search(query, index, num_results=5):
    results = index.search(
    query=query,
    num_results=num_results
    )
    return results

In [28]:
results = search(query, index)
print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [29]:
def build_context(search_results):
    context_parts = []

    for doc in search_results:
        context_parts.append(
            f"""
            FILE: {doc["filename"]}

            CONTENT: {doc["content"]}
            """
        )
    
    return "\n\n".join(context_parts)

In [30]:
context = build_context(results)
print(context[:1000])


            FILE: 01-agentic-rag/lessons/14-agentic-loop.md

            CONTENT: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the instructions, the better the
  agent helps.
- Tools, 

In [31]:
def build_prompt(query, context):
    prompt = f"""
        You are a teaching assistant for the LLM Zoomcamp course.

        Answer the question using only the provided context. 

        QUESTION: {query}
        CONTEXT: {context}
    """

    return prompt

In [32]:
prompt = build_prompt(query, context)
print(prompt)


        You are a teaching assistant for the LLM Zoomcamp course.

        Answer the question using only the provided context. 

        QUESTION: How does the agentic loop keep calling the model until it stops?
        CONTEXT: 
            FILE: 01-agentic-rag/lessons/14-agentic-loop.md

            CONTENT: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an 

In [33]:
from openai import OpenAI

client = OpenAI()

In [34]:
def ask_llm(prompt, client):
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

    return response

In [35]:
def rag(query, index, client):
    search_results = search(query, index)

    context = build_context(search_results)

    prompt = build_prompt(query, context)

    response = ask_llm(prompt, client)

    return response

In [36]:
query = "How does the agentic loop keep calling the model until it stops?"

response = rag(query, index, client)

print(response.output_text)

The loop keeps calling the model by checking whether the response contains any `function_call` items.

- It sends the current `messages` to the model.
- It appends the model’s output to `messages`.
- If the model asks for a tool, the code runs the tool and appends the tool output too.
- Then it sends the updated `messages` back to the model again.

This repeats inside a `while True` loop.

The loop stops when `has_function_calls` is `False`, meaning the model returned a response with no function calls. That means the model has given its final answer, so the code breaks out of the loop.


In [37]:
print(response.usage)

ResponseUsage(input_tokens=7133, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=137, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7270)


In [38]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

print(len(chunks))

295


In [39]:
print(chunks[0].keys())
print(chunks[0]["filename"])
print(chunks[0]["start"])
print(len(chunks[0]["content"]))

dict_keys(['start', 'content', 'filename'])
01-agentic-rag/lessons/01-intro.md
0
2000


In [40]:
print(len(chunks))

295


In [41]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [42]:
chunk_response = rag(query, chunk_index, client)

In [43]:
print(chunk_response.usage)

ResponseUsage(input_tokens=2316, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2430)


In [57]:
search_count = 0

def search(query: str) -> str:
    global search_count

    search_count += 1

    print(f"SEARCH #{search_count}: {query}")

    results = chunk_index.search(
        query=query,
        num_results=5
    )

    return str(results)


In [58]:
print(search("agentic loop"))
print(search_count)

SEARCH #1: agentic loop
[{'start': 4000, 'content': 'esult.\n\nThe `result` is a `LoopResult` with `all_messages` (the full\nconversation), token counts, and `cost` (computed from token usage).\n\n## Cost and tokens\n\nLook at what the call cost:\n\n```python\nresult.cost\n```\n\nUseful while developing - especially with multi-turn agents where one\nprompt can trigger several model calls. The handwritten loop made you\ncompute this by hand. The framework keeps a running total for you.\n\nYou can also look at the full message history.\n\n```python\nresult.all_messages\n```\n\nThis is just a list - the same `messages` list we maintained by hand.\n\n## Continuing the conversation\n\nTake the messages from the previous result and pass them as\n`previous_messages` on the next `loop` call:\n\n```python\nresult2 = runner.loop(\n    prompt="How do I run a different model?",\n    previous_messages=result.all_messages,\n    callback=callback,\n)\n```\n\nThe runner picks up where the last call le

In [59]:
from toyaikit.tools import Tools

agent_tools = Tools()

agent_tools.add_tool(search)

In [60]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [61]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import (
    OpenAIResponsesRunner,
    DisplayingRunnerCallback
)

In [62]:
chat_interface = IPythonChatInterface()

callback = DisplayingRunnerCallback(
    chat_interface
)

In [63]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt="""
You're a course teaching assistant.

Answer the student's question using the search tool.

Make multiple searches with different keywords before answering.
""",
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model="gpt-5.4-mini"
    )
)

In [64]:
print(type(runner))

<class 'toyaikit.chat.runners.OpenAIResponsesRunner'>


In [65]:
search_count = 0

In [66]:
result = runner.loop(
    prompt="""
How does the agentic loop work,
and how is it different from plain RAG?
""",
    callback=callback,
)

-> Response received
SEARCH #1: agentic loop compared to RAG definition


SEARCH #2: agentic loop in LLM systems iterative planning tool use


SEARCH #3: plain RAG vs agentic RAG difference


SEARCH #4: retrieval augmented generation plain RAG workflow


-> Response received


In [67]:
print(search_count)

4
